In [1]:
from keras.utils import to_categorical, load_img
from keras.models import Sequential
from keras.layers import Dense, Conv2D, Dropout, Flatten, MaxPooling2D
import os
import pandas as pd
import numpy as np

In [2]:
TRAIN_DIR = 'images/train'
TEST_DIR = 'images/test'

In [3]:
def createdataframe(dir):
    image_paths = []
    labels = []
    for label in os.listdir(dir):
        for imagename in os.listdir(os.path.join(dir,label)):
            image_paths.append(os.path.join(dir,label,imagename))
            labels.append(label)
        print(label, "completed")
    return image_paths,labels

In [4]:
train = pd.DataFrame()
train['image'], train['label'] = createdataframe(TRAIN_DIR)

angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed


In [5]:
print(train)

                                image     label
0            images/train\angry\0.jpg     angry
1            images/train\angry\1.jpg     angry
2           images/train\angry\10.jpg     angry
3        images/train\angry\10002.jpg     angry
4        images/train\angry\10016.jpg     angry
...                               ...       ...
28816  images/train\surprise\9969.jpg  surprise
28817  images/train\surprise\9985.jpg  surprise
28818  images/train\surprise\9990.jpg  surprise
28819  images/train\surprise\9992.jpg  surprise
28820  images/train\surprise\9996.jpg  surprise

[28821 rows x 2 columns]


In [6]:
test = pd.DataFrame()
test['image'], test['label'] = createdataframe(TEST_DIR)

angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed


In [7]:
print(test)

                              image     label
0       images/test\angry\10052.jpg     angry
1       images/test\angry\10065.jpg     angry
2       images/test\angry\10079.jpg     angry
3       images/test\angry\10095.jpg     angry
4       images/test\angry\10121.jpg     angry
...                             ...       ...
7061  images/test\surprise\9806.jpg  surprise
7062  images/test\surprise\9830.jpg  surprise
7063  images/test\surprise\9853.jpg  surprise
7064  images/test\surprise\9878.jpg  surprise
7065   images/test\surprise\993.jpg  surprise

[7066 rows x 2 columns]


In [8]:
from tqdm.notebook import tqdm

In [9]:
def extract_features(images):
    features = []
    for image in tqdm(images):
        img = load_img(image, color_mode="grayscale")
        img = np.array(img)
        features.append(img)
    features = np.array(features)
    features = features.reshape(len(features),48,48,1)
    return features

In [10]:
train_features = extract_features(train['image'])

  0%|          | 0/28821 [00:00<?, ?it/s]

In [11]:
train_features.shape

(28821, 48, 48, 1)

In [12]:
test_features = extract_features(test['image'])

  0%|          | 0/7066 [00:00<?, ?it/s]

In [13]:
x_train = train_features/255.0
x_test = test_features/255.0

In [14]:
from sklearn.preprocessing import LabelEncoder

In [15]:
le = LabelEncoder()
le.fit(train['label'])

Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)Holds the label for each class.","ndarray[object](7,)","['angry','disgust','fear',...,'neutral','sad','surprise']"


In [16]:
y_train = le.transform(train['label'])
y_test = le.transform(test['label'])

In [17]:
y_train = to_categorical(y_train,num_classes = 7)
y_test = to_categorical(y_test,num_classes = 7)

In [18]:
from keras.layers import Input

model = Sequential()
model.add(Input(shape=(48, 48, 1)))

# convolutional layers
model.add(Conv2D(128, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(256, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(512, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(512, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Flatten())

# fully connected layers
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.3))

# output layer
model.add(Dense(7, activation='softmax'))


In [21]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [22]:
epochs=5

In [23]:
model.fit(x= x_train, y = y_train, batch_size = 128, epochs = 100, validation_data = (x_test,y_test))

Epoch 1/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 210s 910ms/step - accuracy: 0.2427 - loss: 1.8251 - val_accuracy: 0.2583 - val_loss: 1.8081
Epoch 2/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 189s 836ms/step - accuracy: 0.2557 - loss: 1.7919 - val_accuracy: 0.2887 - val_loss: 1.7251
Epoch 3/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 189s 835ms/step - accuracy: 0.3074 - loss: 1.7027 - val_accuracy: 0.3670 - val_loss: 1.6135
Epoch 4/100
116/226 ━━━━━━━━━━━━━━━━━━━━ 1:29 815ms/step - accuracy: 0.3564 - loss: 1.6176

KeyboardInterrupt: 

In [24]:
from keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ModelCheckpoint('emotion_model.keras', monitor='val_accuracy', save_best_only=True)
]

model.fit(x=x_train, y=y_train, batch_size=128, epochs=100,
          validation_data=(x_test, y_test), callbacks=callbacks)

Epoch 1/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 185s 818ms/step - accuracy: 0.4060 - loss: 1.5249 - val_accuracy: 0.4691 - val_loss: 1.3846
Epoch 2/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 191s 843ms/step - accuracy: 0.4354 - loss: 1.4597 - val_accuracy: 0.4844 - val_loss: 1.3576
Epoch 3/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 184s 816ms/step - accuracy: 0.4517 - loss: 1.4142 - val_accuracy: 0.5038 - val_loss: 1.3075
Epoch 4/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 181s 800ms/step - accuracy: 0.4704 - loss: 1.3700 - val_accuracy: 0.5195 - val_loss: 1.2566
Epoch 5/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 196s 868ms/step - accuracy: 0.4838 - loss: 1.3430 - val_accuracy: 0.5134 - val_loss: 1.2441
Epoch 6/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 180s 795ms/step - accuracy: 0.5009 - loss: 1.3135 - val_accuracy: 0.5403 - val_loss: 1.2087
Epoch 7/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 179s 790ms/step - accuracy: 0.5049 - loss: 1.2941 - val_accuracy: 0.5440 - val_loss: 1.2046
Epoch 8/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 181s 803ms/step - accuracy: 

In [27]:
from keras.models import load_model
model = load_model("emotion_model.keras")

In [31]:
from keras.models import load_model
from keras.utils import load_img
import numpy as np

model = load_model("emotion_model.keras")
labels = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

def predict_emotion(path):
    img = load_img(path, color_mode="grayscale", target_size=(48, 48))
    img = np.array(img).reshape(1, 48, 48, 1) / 255.0
    pred = model.predict(img, verbose=0)
    return labels[pred.argmax()]

# try an image from your test folder
path = os.path.join(TEST_DIR, "happy", os.listdir(os.path.join(TEST_DIR, "happy"))[0])
print("Actual: happy | Predicted:", predict_emotion(path))

Actual: happy | Predicted: sad
